# Проверка экспериментальных результатов НИР-3
Ноутбук использует сохраненные численные результаты унифицированной контрольной серии. Запуск обучения выполняется отдельно скриптом `run_unified_24_cv.py`.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score
import numpy as np
ROOT = Path.cwd()
RESULTS = ROOT / 'results' if (ROOT/'results').exists() else Path('Source_Code/results')
test = pd.read_csv(RESULTS/'test_and_cv_summary.csv')
folds = pd.read_csv(RESULTS/'cv_5fold_metrics.csv')
ci = pd.read_csv(RESULTS/'unified_bootstrap_intervals_5000.csv')
assert len(test)==24 and len(folds)==120
test[['dataset','model','treatment','test_AP','cv_AP_mean']].head()


## Сопоставление моделей по AP
График создается из результатов; исходные значения не подменяются вручную.


In [ ]:
labels = test['dataset'] + ' / ' + test['model']
pivot = test.assign(label=labels).pivot(index='label', columns='treatment', values='test_AP')
pivot = pivot[['baseline','weighting','ros','smote']]
ax = pivot.plot.bar(figsize=(11,5), ylabel='Average Precision', xlabel='', rot=35)
ax.legend(title='Режим', loc='lower right')
plt.tight_layout()
plt.show()


## Контроль значения AP по тестовым вероятностям
Для каждого набора и конфигурации вычисляется та же AP, что сохранена в CSV.


In [ ]:
arr = np.load(RESULTS/'test_predictions.npz')
for r in test.itertuples():
    y = arr[f'{r.dataset}_y']
    prob = arr[f'{r.dataset}_{r.model}_{r.treatment}']
    assert abs(average_precision_score(y, prob)-r.test_AP) < 1e-10
print('Все 24 AP совпадают')
ci[['metric','point_estimate','CI95_lower','CI95_upper']].head(5)


## Статистическое разграничение
Историческая подтверждающая H1 и эта унифицированная серия имеют разные настройки обучения. Показанные интервалы отражают вариацию фиксированных прогнозов, а не повторных обучений.
